# misc things like verifying functions

In [10]:
?verify_function

Object `verify_function` not found.


# Exporting Models

We are gonna train a small model from [this tutorial](https://www.tanishq.ai/blog/posts/2021-11-16-gradio-huggingface.html).

Using a simple pet classifier with [ResNet50](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet50.html).

In [12]:
from fastai.vision.all import *
path = untar_data(URLs.PETS)
dls = ImageDataLoaders.from_name_re(path, get_image_files(path/'images'), pat='(.+)_\d+.jpg', item_tfms=Resize(448), batch_tfms=aug_transforms(size=224, min_scale=0.75))
learn = vision_learner(dls, models.resnet50, metrics=accuracy)
learn.fine_tune(1)
learn.path = Path('.')
learn.export()

epoch,train_loss,valid_loss,accuracy,time
0,1.437504,0.364406,0.888363,00:59


epoch,train_loss,valid_loss,accuracy,time
0,0.434801,0.254631,0.920839,01:10


# Using Gradio

Now that we have an exported model, we can load it like this

In [16]:
learn = load_learner('export.pkl')

And now we can create a `predict` function that can be called by other functions.

In [17]:
labels = learn.dls.vocab
def predict(img):
    img = PILImage.create(img)
    pred,pred_idx,probs = learn.predict(img)
    return {labels[i]: float(probs[i]) for i in range(len(labels))}

Now we import gradio and make it generate an UI for us.

In [ ]:
import gradio as gr

gr.Interface(
    fn=predict,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=3)
).launch(share=True)

This generates a gradio model locally. Which is what we need. To have all code in one single python block to export to hugging face we can do something like this:

In [ ]:
# app.py
import gradio as gr
from fastai.vision.all import *
import skimage

learn = load_learner('export.pkl')

labels = learn.dls.vocab
def predict(img):
    img = PILImage.create(img)
    pred,pred_idx,probs = learn.predict(img)
    return {labels[i]: float(probs[i]) for i in range(len(labels))}

title = "Pet Breed Classifier"
description = "A pet breed classifier trained on the Oxford Pets dataset with fastai. Created as a demo for Gradio and HuggingFace Spaces."
examples = ['siamese.jpg'] # we need an image as example here
article = """<div style="margin-top: 12px; font-size: 14px; color: #666; text-align: center;">
made by <a href="https://luskira.com" style="text-decoration: underline; color: white; display: inline-flex; align-items: center; gap: 4px;">luskira<img src="https://lucas-schiavini.com/content/images/size/w320/2025/03/New-pfp--2--1.png" alt="luskira" style="width: 24px; height: 24px; border-radius: 50%; object-fit: cover;" /></a>
</div>"""

gr.Interface(
    fn=predict,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=3),
    title=title,
    description=description,
    article=article,
    examples=examples,
).launch(share=True)

And all we need is the export.pkl model to import anywhere, like on a hugging face project. We don't need to train models all the time. Just once and then export.

My project currently sits at https://huggingface.co/spaces/luskira/minimal.